# Week 03 — Camera and Gallery Image Processing
## LeafGuard AI: Preparing Images for Diagnosis

This notebook mirrors the image-processing ideas behind Android camera and gallery flows, but uses Python to explore them safely and quickly.

**Learning outcomes:**
- Load and resize images with Pillow
- Understand the `inSampleSize` downscaling idea from Android
- Simulate EXIF orientation correction
- Inspect colours from bitmap pixels
- Model runtime permission decisions as a state machine

---


## Cell 1: Create and Resize a Demo Image

Pillow is the Python equivalent of many bitmap operations you perform on Android.


In [ ]:
# If Pillow is missing, install it with: pip install Pillow
from PIL import Image, ImageDraw

image = Image.new('RGB', (640, 480), color=(34, 139, 34))
draw = ImageDraw.Draw(image)
draw.ellipse((180, 80, 460, 360), fill=(144, 238, 144), outline=(255, 255, 255), width=6)

resized = image.resize((224, 224))
print(f'Original size: {image.size}')
print(f'Resized size:  {resized.size}')
print(f'Mode: {resized.mode}')


## Cell 2: Recreate Android's `inSampleSize` Idea

Android often downsamples large images before decoding them fully to save memory.


In [ ]:
def calculate_in_sample_size(original_width, original_height, req_width, req_height):
    in_sample_size = 1
    while (original_height // (in_sample_size * 2) >= req_height and
           original_width // (in_sample_size * 2) >= req_width):
        in_sample_size *= 2
    return in_sample_size

sample_size = calculate_in_sample_size(4032, 3024, 224, 224)
print('=== INSAMPLESIZE DEMO ===')
print(f'Camera image: 4032x3024')
print(f'Target size:  224x224')
print(f'Chosen inSampleSize: {sample_size}')
print(f'Approx decoded size: {4032 // sample_size}x{3024 // sample_size}')


## Cell 3: Simulate EXIF Orientation Correction

Camera images may look rotated because the sensor orientation is stored as metadata. The app must correct that before inference.


In [ ]:
orientation_map = {
    1: 'Normal',
    3: 'Rotate 180°',
    6: 'Rotate 270° clockwise',
    8: 'Rotate 90° clockwise',
}

exif_orientation = 6
corrected = image.transpose(Image.Transpose.ROTATE_270) if exif_orientation == 6 else image

print('=== EXIF ORIENTATION ===')
print(f'EXIF orientation code: {exif_orientation}')
print(f'Action: {orientation_map[exif_orientation]}')
print(f'Corrected image size: {corrected.size}')


## Cell 4: Sample Pixel Colours from the Bitmap

Pixel inspection is useful for debugging preprocessing and checking whether the image content looks plausible.


In [ ]:
sample_points = [(10, 10), (112, 112), (180, 120)]
print('=== PIXEL SAMPLES ===')
for point in sample_points:
    print(f'{point} -> {resized.getpixel(point)}')


## Cell 5: Compute a Simple Average Colour

A coarse average colour is not enough for diagnosis, but it is a helpful sanity check during image processing.


In [ ]:
pixels = list(resized.getdata())
avg_r = sum(p[0] for p in pixels) / len(pixels)
avg_g = sum(p[1] for p in pixels) / len(pixels)
avg_b = sum(p[2] for p in pixels) / len(pixels)

print('=== AVERAGE COLOUR ===')
print(f'R={avg_r:.2f}, G={avg_g:.2f}, B={avg_b:.2f}')


## Cell 6: Simulate Runtime Permission Handling

Camera and gallery features rely on runtime permissions. State-machine thinking makes those flows easier to implement and debug.


In [ ]:
transitions = [
    ('Unknown', 'request_permission', 'PendingUserChoice'),
    ('PendingUserChoice', 'user_denies', 'Denied'),
    ('Denied', 'show_rationale', 'NeedsExplanation'),
    ('NeedsExplanation', 'request_again', 'PendingUserChoice'),
    ('PendingUserChoice', 'user_grants', 'Granted'),
    ('Granted', 'open_camera', 'CameraReady'),
]

print('=== PERMISSION STATE MACHINE ===')
for current, action, next_state in transitions:
    print(f'{current:<18} --{action:<16}--> {next_state}')


## Summary and Quick Quiz

**Key takeaways:**
- Image resizing, orientation correction, and permission handling are all part of a reliable camera workflow.
- Memory-friendly decoding is essential when working with large phone images.
- Simple pixel checks can reveal preprocessing bugs early.

**Quick quiz:**
1. Why is `inSampleSize` usually a power of two?
2. What happens if EXIF orientation is ignored?
3. Why should permissions be modeled as states instead of one-off booleans?
4. How could bad resizing hurt model accuracy?


## Parallel Kotlin Track — Week 03: Camera & Gallery in Kotlin

All of the image-processing concepts above (resizing, orientation, permissions) are language-neutral.
Here is how the capture flow from the Java app translates to the Kotlin twin.

### Java vs Kotlin: registering ActivityResult launchers

Java (`android-app/.../MainActivity.java`):

```java
cameraLauncher = registerForActivityResult(
        new ActivityResultContracts.TakePicture(),
        success -> {
            if (success && pendingCameraUri != null) {
                updateSelectedImage(pendingCameraUri);
            }
        }
);
```

Kotlin (`android-app-kotlin/.../MainActivity.kt`):

```kotlin
cameraLauncher = registerForActivityResult(
    ActivityResultContracts.TakePicture()
) { success ->
    val cameraUri = pendingCameraUri
    if (success && cameraUri != null) {
        updateSelectedImage(cameraUri)
    }
}
```

### Permission checks with Kotlin collection functions

```kotlin
private fun hasPermissions(permissions: Array<String>): Boolean {
    return permissions.all { permission ->
        ContextCompat.checkSelfPermission(this, permission) == PackageManager.PERMISSION_GRANTED
    }
}
```

The Java version needs an explicit `for` loop with an early `return false`; Kotlin's `all { }` expresses
the same logic in one line — same behavior, same permission strings (`CAMERA`, `READ_MEDIA_IMAGES` on
API 33+, `READ_EXTERNAL_STORAGE` before).

### Try this
Compare `requiredGalleryPermissions()` in both `MainActivity.java` and `MainActivity.kt` — confirm the
API-level branching (`Build.VERSION_CODES.TIRAMISU`) is identical.

**Expected output:** identical permission dialogs and camera/gallery flows in both apps.

**✅ Checkpoint:** why must the FileProvider authority (`${applicationId}.fileprovider`) be the same in both manifests?

**⚠️ Common mistake:** capturing `pendingCameraUri` directly in the lambda without a local `val` — Kotlin smart-casts only stable values.

**🔑 Key point:** ActivityResult contracts, FileProvider, and permission flows are API-identical across languages; Kotlin just removes boilerplate.